In [1]:
import math

class Node:
    def __init__(self, name, value=None):
        self.name = name
        self.value = value
        self.children = []
        self.minmax_value = None


class Environment:
    def compute_minimax(self, node, maximizing_player):
        # Leaf node
        if not node.children:
            return node.value

        if maximizing_player:
            value = -math.inf
            for child in node.children:
                value = max(value, self.compute_minimax(child, False))
            node.minmax_value = value
            return value

        else:
            value = math.inf
            for child in node.children:
                value = min(value, self.compute_minimax(child, True))
            node.minmax_value = value
            return value


class Agent:
    def act(self, root, env):
        print("Building tree...\n")

        result = env.compute_minimax(root, True)

        print("\nMinimax computation complete\n")
        print("Final decision value:", result)

        # Show states
        print("\nStates:")
        print(f"{root.name}: {root.minmax_value}")
        for c in root.children:
            print(f"{c.name}: {c.minmax_value}")


# -------------------------
# BUILD TREE
# -------------------------

root = Node("Robot")

zoneA = Node("Zone A")
zoneB = Node("Zone B")

# Thief choices
zoneA.children = [Node("A1", 3), Node("A2", 5)]
zoneB.children = [Node("B1", 2), Node("B2", 9)]

root.children = [zoneA, zoneB]

# Run
env = Environment()
agent = Agent()
agent.act(root, env)

Building tree...


Minimax computation complete

Final decision value: 3

States:
Robot: 3
Zone A: 3
Zone B: 2


In [4]:
import math

class AlphaBetaTracer:
    def __init__(self, tree):
        self.tree = tree
        self.step = 0

    def log(self, message):
        print(message)

    def alphabeta(self, node, depth, alpha, beta, maximizing, path):
        indent = "    " * len(path)

        # Leaf node
        if isinstance(node, int):
            self.step += 1
            self.log(f"{indent}Leaf {node} evaluated. [{alpha}, {beta}]")
            return node

        self.log(f"{indent}{'MAX' if maximizing else 'MIN'} at {node['name']} starts. [{alpha}, {beta}]")

        if maximizing:
            value = -math.inf

            for i, child in enumerate(node["children"]):
                self.log(f"{indent} Exploring {node['name']} option {i+1}")

                val = self.alphabeta(child, depth + 1, alpha, beta, False, path + [node['name']])

                value = max(value, val)
                alpha = max(alpha, value)

                self.log(f"{indent} Updated MAX value at {node['name']} = {value}. [{alpha}, {beta}]")

                if beta <= alpha:
                    self.log(f"{indent} PRUNING at {node['name']} (beta <= alpha)")
                    break

            return value

        else:
            value = math.inf

            for i, child in enumerate(node["children"]):
                self.log(f"{indent} Exploring {node['name']} response {i+1}")

                val = self.alphabeta(child, depth + 1, alpha, beta, True, path + [node['name']])

                value = min(value, val)
                beta = min(beta, value)

                self.log(f"{indent} Updated MIN value at {node['name']} = {value}. [{alpha}, {beta}]")

                if beta <= alpha:
                    self.log(f"{indent} PRUNING at {node['name']} (beta <= alpha)")
                    break

            return value


# -------------------------
# TREE STRUCTURE (TASK 2)
# -------------------------

tree = {
    "name": "Root",
    "children": [
        {
            "name": "Stop",
            "children": [6, 7]
        },
        {
            "name": "Go",
            "children": [2, 4]
        },
        {
            "name": "Turn",
            "children": [8, 3]
        }
    ]
}

# Run
tracer = AlphaBetaTracer(tree)
result = tracer.alphabeta(tree, 0, -math.inf, math.inf, True, [])

print("\nFINAL RESULT:", result)

print("\n================ TASK 2 (C) & (D) =================")

print("\n(c) Pruned Branches:")
print("- In 'Go' branch, the second leaf node (value 4) was pruned due to alpha-beta cutoff.")
print("- This happened because beta <= alpha condition was satisfied early.")

print("\n(d) Nodes not evaluated due to pruning:")
print("Total leaf nodes: 6")
print("Evaluated nodes: 5")
print("Pruned nodes: 1")

MAX at Root starts. [-inf, inf]
 Exploring Root option 1
    MIN at Stop starts. [-inf, inf]
     Exploring Stop response 1
        Leaf 6 evaluated. [-inf, inf]
     Updated MIN value at Stop = 6. [-inf, 6]
     Exploring Stop response 2
        Leaf 7 evaluated. [-inf, 6]
     Updated MIN value at Stop = 6. [-inf, 6]
 Updated MAX value at Root = 6. [6, inf]
 Exploring Root option 2
    MIN at Go starts. [6, inf]
     Exploring Go response 1
        Leaf 2 evaluated. [6, inf]
     Updated MIN value at Go = 2. [6, 2]
     PRUNING at Go (beta <= alpha)
 Updated MAX value at Root = 6. [6, inf]
 Exploring Root option 3
    MIN at Turn starts. [6, inf]
     Exploring Turn response 1
        Leaf 8 evaluated. [6, inf]
     Updated MIN value at Turn = 8. [6, 8]
     Exploring Turn response 2
        Leaf 3 evaluated. [6, 8]
     Updated MIN value at Turn = 3. [6, 3]
     PRUNING at Turn (beta <= alpha)
 Updated MAX value at Root = 6. [6, inf]

FINAL RESULT: 6

================ TASK 2 (C) & (

In [9]:
import math

class Task3Tracer:
    def log(self, msg):
        print(msg)

    def alphabeta(self, node, alpha, beta, maximizing, path):

        indent = "    " * len(path)

        if isinstance(node, int):
            self.log(f"{indent}Leaf {node} evaluated. [{alpha}, {beta}]")
            return node

        self.log(f"{indent}{'MAX' if maximizing else 'MIN'} at {node['name']} starts. [{alpha}, {beta}]")

        if maximizing:
            value = -math.inf

            for child in node["children"]:
                child_name = child["name"] if isinstance(child, dict) else str(child)

                self.log(f"{indent} Exploring {node['name']} → {child_name}")

                val = self.alphabeta(child, alpha, beta, False, path + [node["name"]])

                value = max(value, val)
                alpha = max(alpha, value)

                self.log(f"{indent} Updated MAX at {node['name']} = {value}. [{alpha}, {beta}]")

                if beta <= alpha:
                    self.log(f"{indent} PRUNED at {node['name']}")
                    break

            return value

        else:
            value = math.inf

            for child in node["children"]:
                child_name = child["name"] if isinstance(child, dict) else str(child)

                self.log(f"{indent} Exploring {node['name']} → {child_name}")

                val = self.alphabeta(child, alpha, beta, True, path + [node["name"]])

                value = min(value, val)
                beta = min(beta, value)

                self.log(f"{indent} Updated MIN at {node['name']} = {value}. [{alpha}, {beta}]")

                if beta <= alpha:
                    self.log(f"{indent} PRUNED at {node['name']}")
                    break

            return value


tree = {
    "name": "Root",
    "children": [
        {
            "name": "Attack",
            "children": [
                {"name": "A1", "children": [5, 6]},
                {"name": "A2", "children": [7, 4]}
            ]
        },
        {
            "name": "Defend",
            "children": [
                {"name": "D1", "children": [3, 8]},
                {"name": "D2", "children": [6, 2]}
            ]
        },
        {
            "name": "Gather",
            "children": [
                {"name": "G1", "children": [1, 9]},
                {"name": "G2", "children": [4, 7]}
            ]
        }
    ]
}

# RUN
tracer = Task3Tracer()
result = tracer.alphabeta(tree, -math.inf, math.inf, True, [])

print("\nFINAL RESULT:", result)

print("\n================ TASK 3 (C), (D) & (E) =================")

print("\n(c) Pruned Branches:")

print("- In 'Attack' branch, some deeper evaluations under A2 may be pruned if alpha >= beta condition is met early.")
print("- In 'Defend' branch, D2 subtree may be partially or fully pruned depending on cutoff point.")
print("- In 'Gather' branch, G2 subtree may be skipped in some paths due to alpha-beta cutoff.")
print("- Pruning occurs whenever beta <= alpha condition becomes true during traversal.")

print("\n(d) Comparison of Nodes Evaluated:")

print("Minimax Algorithm:")
print("- Evaluates all leaf nodes in the tree")
print("- Total leaf nodes = 12")
print("- No pruning applied")

print("\nAlpha-Beta Pruning:")
print("- Evaluates fewer nodes due to pruning")
print("- Some leaf nodes are skipped during traversal")
print("- Exact number depends on traversal order, but always <= Minimax")

print("\nConclusion:")
print("- Alpha-Beta Pruning reduces search space while preserving optimal result")

print("\n(e) Efficiency Improvement:")

print("- Alpha-Beta Pruning improves efficiency by eliminating unnecessary branches")
print("- It avoids exploring paths that cannot affect the final decision")
print("- It reduces computation time compared to Minimax")
print("- It is especially useful in large game trees with high branching factor")

print("\nFinal Insight:")
print("- Both Minimax and Alpha-Beta produce the same optimal decision")
print("- Alpha-Beta is faster due to reduced node exploration")

MAX at Root starts. [-inf, inf]
 Exploring Root → Attack
    MIN at Attack starts. [-inf, inf]
     Exploring Attack → A1
        MAX at A1 starts. [-inf, inf]
         Exploring A1 → 5
            Leaf 5 evaluated. [-inf, inf]
         Updated MAX at A1 = 5. [5, inf]
         Exploring A1 → 6
            Leaf 6 evaluated. [5, inf]
         Updated MAX at A1 = 6. [6, inf]
     Updated MIN at Attack = 6. [-inf, 6]
     Exploring Attack → A2
        MAX at A2 starts. [-inf, 6]
         Exploring A2 → 7
            Leaf 7 evaluated. [-inf, 6]
         Updated MAX at A2 = 7. [7, 6]
         PRUNED at A2
     Updated MIN at Attack = 6. [-inf, 6]
 Updated MAX at Root = 6. [6, inf]
 Exploring Root → Defend
    MIN at Defend starts. [6, inf]
     Exploring Defend → D1
        MAX at D1 starts. [6, inf]
         Exploring D1 → 3
            Leaf 3 evaluated. [6, inf]
         Updated MAX at D1 = 3. [6, inf]
         Exploring D1 → 8
            Leaf 8 evaluated. [6, inf]
         Updated MAX at